In [ ]:
# Feature Engineering

## Objective

#Raw interactions such as views, clicks, and cart additions do not directly represent user preferences.

#To build a recommendation system, we need to transform these actions into numerical signals.

#This notebook creates:

#- Interaction Scores
#- Engagement Scores
#- User-Product Preference Table

#These engineered features will later be used for collaborative filtering.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving eda_checked.csv to eda_checked.csv


In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('eda_checked.csv')

In [4]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 100000
Columns: 7


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   interaction_id    100000 non-null  object
 1   user_id           100000 non-null  object
 2   product_id        100000 non-null  object
 3   session_id        100000 non-null  object
 4   interaction_type  100000 non-null  object
 5   timestamp         100000 non-null  object
 6   dwell_time_ms     100000 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [6]:
df["interaction_type"].value_counts()

,count
interaction_type,
view,50463
click,19983
add_to_cart,11860
add_to_wishlist,10168
remove_from_wishlist,4795
remove_from_cart,2731


In [7]:
weights = {
    "view": 1,
    "click": 3,
    "add_to_wishlist": 5,
    "add_to_cart": 8,
    "remove_from_wishlist": -2,
    "remove_from_cart": -3
}

In [8]:
def create_interaction_scores(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Convert interaction types
    into numerical scores.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    engineered_df = df.copy()

    engineered_df["interaction_score"] = (
        engineered_df["interaction_type"]
        .map(weights)
    )

    return engineered_df

In [9]:
df = create_interaction_scores(df)

In [10]:
df[
    [
        "interaction_type",
        "interaction_score"
    ]
].head(20)

,interaction_type,interaction_score
0,view,1
1,view,1
2,view,1
3,view,1
4,add_to_wishlist,5
5,remove_from_wishlist,-2
6,view,1
7,view,1
8,click,3
9,add_to_cart,8


In [11]:
df["interaction_score"].value_counts()

,count
interaction_score,
1,50463
3,19983
8,11860
5,10168
-2,4795
-3,2731


In [12]:
df["dwell_time_seconds"] = (
    df["dwell_time_ms"] / 1000
)

In [13]:
df["dwell_time_seconds"].describe()

,dwell_time_seconds
count,100000.000000
mean,17.942901
std,21.556684
min,0.500000
25%,4.260000
50%,11.153000
75%,23.342250
max,300.000000


In [14]:
df["engagement_score"] = (
    df["interaction_score"]
    *
    np.log1p(df["dwell_time_seconds"])
)

In [15]:
df[
    [
        "interaction_type",
        "interaction_score",
        "dwell_time_seconds",
        "engagement_score"
    ]
].head()

,interaction_type,interaction_score,dwell_time_seconds,engagement_score
0,view,1,30.481,3.449384
1,view,1,37.889,3.660711
2,view,1,11.801,2.549523
3,view,1,11.801,2.549523
4,add_to_wishlist,5,1.404,4.385670


In [16]:
user_product_preferences = (
    df
    .groupby(
        ["user_id", "product_id"]
    )
    .agg(
        total_score=(
            "interaction_score",
            "sum"
        ),

        total_engagement=(
            "engagement_score",
            "sum"
        ),

        interaction_count=(
            "interaction_type",
            "count"
        )
    )
    .reset_index()
)

In [17]:
user_product_preferences.head()

,user_id,product_id,total_score,total_engagement,interaction_count
0,0000780a-2126-4e84-9622-42ce0ea9b17a,26fa0037-c745-4789-8a44-9f3dfb7f25f7,5,8.471652,1
1,0000780a-2126-4e84-9622-42ce0ea9b17a,5724e855-5b64-43be-ae0d-aa26623226da,6,10.369723,3
2,0000780a-2126-4e84-9622-42ce0ea9b17a,5ad1dceb-cb0e-4de9-89e6-f7c4c6abda23,3,4.006582,1
3,0000780a-2126-4e84-9622-42ce0ea9b17a,5d07bcd4-3670-4a0d-91f6-e36d7d335597,3,10.000144,2
4,0000780a-2126-4e84-9622-42ce0ea9b17a,5d836ef6-20f0-47c1-9a54-791a4bfeded1,1,3.545500,1


In [18]:
user_product_preferences.sort_values(
    by="total_score",
    ascending=False
).head(20)

,user_id,product_id,total_score,total_engagement,interaction_count
34443,a62fcf77-10b9-418b-aadc-e90e9d506e49,25a6c99c-d63c-4697-91ef-744b7879d605,119,317.805493,45
29844,8fd9f351-43b0-4908-a0c1-1a401cf86e46,58fd8edb-60ef-4856-bf1f-644ffc133d12,108,280.477618,44
4386,158ae1ce-505c-4b5e-9d12-83c596f94ba0,6e9b539a-77ad-4307-acb5-f558f8b5cc6f,102,215.940766,36
44716,d627150f-4655-4063-8880-97f6291842f2,58fd8edb-60ef-4856-bf1f-644ffc133d12,98,256.115840,46
40767,c30e5a9d-6b74-46ac-a71e-ec3f98c0749f,c1ab2206-9b50-4aaa-9446-f322d0976ea6,96,223.583246,28
2434,0c19d6da-9401-45f3-a632-3f4c09ff10a3,2e1a3d22-2a05-4b6e-bfd8-4b5d5dab3009,92,238.758195,37
43180,cf2538fb-6067-4bad-99e0-4377f3e1e031,eaaf2b74-b09a-4bf5-b0b2-09021044cab8,90,236.590319,30
13526,413f3f0b-1b60-4782-a10b-bc6a42a89233,6e9b539a-77ad-4307-acb5-f558f8b5cc6f,88,220.494903,35
42378,cb632616-656e-4cbf-968d-bcce3d2c4781,2e1a3d22-2a05-4b6e-bfd8-4b5d5dab3009,83,195.029090,27
51859,f7bdacd9-e92d-45dd-aed0-2654aec08183,6e9b539a-77ad-4307-acb5-f558f8b5cc6f,82,197.364342,34


In [19]:
print(
    "Unique Users:",
    user_product_preferences["user_id"].nunique()
)

print(
    "Unique Products:",
    user_product_preferences["product_id"].nunique()
)

print(
    "User Product Pairs:",
    len(user_product_preferences)
)

Unique Users: 6944
Unique Products: 967
User Product Pairs: 53796


In [21]:
user_product_preferences.to_csv('user_product_preferences.csv', index=False)
from google.colab import files
files.download('user_product_preferences.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>